<a href="https://colab.research.google.com/github/sampabiet90/AIML/blob/LogicMojo-AI-ML-April26-sampa90/tropical_flower_variety_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
from datasets import load_dataset
from copy import deepcopy
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights



In [29]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)

Device: cpu


In [30]:
ds = load_dataset("Project-AgML/tropical_flower_variety_classification")

In [31]:
print(ds["train"].features)

{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Bougainvillea', 'Crown of thorns', 'Hibiscus', 'Jungle geranium', 'Madagascar periwinkle', 'Marigold', 'Rose'])}


In [32]:
SEED = 42
split = ds["train"].train_test_split(
    test_size=0.2,
    seed=SEED
)

train_ds = split["train"]
temp_ds = split["test"]

# Split temporary 20% into validation and test
split_test = temp_ds.train_test_split(
    test_size=0.5,
    seed=SEED
)

valid_ds = split_test["train"]
test_ds = split_test["test"]

print("Training images:", len(train_ds))
print("Validation images:", len(valid_ds))
print("Testing images:", len(test_ds))

Training images: 3455
Validation images: 432
Testing images: 432


In [34]:
## Tranform using Augmentation

train_transform = transforms.Compose([
    transforms.Resize((180, 180)),

    transforms.RandomResizedCrop(
        160,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=15
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    # IMPORTANT: PIL Image -> Tensor
    transforms.ToTensor(),

    # Tensor -> normalized Tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


valid_transform = transforms.Compose([
    transforms.Resize((160, 160)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [35]:
## Pytorch Dataset

class FlowerDataset(Dataset):
  def __init__(self, ds, transform=None):
    self.ds = ds
    self.transform = transform

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, index) :
     ex = self.ds[index]
     image = ex["image"]
     label = ex["label"]
     # convert to RGB
     image = image.convert("RGB")

    # Apply transform only when image is requested
     if self.transform is not None:
          image = self.transform(image)

     return image, torch.tensor(
          label,
          dtype=torch.long
      )


In [36]:
# create train , valid and test test dataset
train_dataset = FlowerDataset(
    train_ds,
    transform=train_transform
)

valid_dataset = FlowerDataset(
    valid_ds,
    transform=valid_transform
)

test_dataset = FlowerDataset(
    test_ds,
    transform=valid_transform
)

In [37]:
# create dataloader
def make_loader(dataset, shuffle=False, seed=SEED):

    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=16,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0
    )

train_loader = make_loader(
    train_dataset,
    shuffle=True,
    seed=SEED
)

valid_loader = make_loader(
    valid_dataset,
    shuffle=False
)

test_loader = make_loader(
    test_dataset,
    shuffle=False
)

In [38]:
# WHAT: Define one training epoch and one deterministic evaluation pass.
# WHY: All strategies should use the same loss and metric calculations.
# OUTPUT: Reusable functions returning loss, accuracy, labels, and predictions.

def train_one_epoch(model, loader, optimizer, *, use_augmentation):
    model.train()

    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, count = 0.0, 0, 0
    for raw_images, labels in loader:

        inputs = raw_images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        count += labels.size(0)
    return loss_sum / count, correct / count

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, count = 0.0, 0
    labels_all, predictions_all = [], []
    for raw_images, labels in loader:
        inputs = raw_images.to(DEVICE)
        labels_device = labels.to(DEVICE)
        logits = model(inputs)
        loss = criterion(logits, labels_device)

        loss_sum += loss.item() * labels.size(0)
        count += labels.size(0)
        labels_all.append(labels)
        predictions_all.append(logits.argmax(1).cpu())

    labels_array = torch.cat(labels_all).numpy()
    predictions_array = torch.cat(predictions_all).numpy()
    return {
        "loss": loss_sum / count,
        "accuracy": float((labels_array == predictions_array).mean()),
        "labels": labels_array,
        "predictions": predictions_array,
    }

In [39]:
# WHAT: Train a model, keep its best validation state, and stop when improvement stalls.
# WHY: The last epoch is not automatically the best model.
# OUTPUT: Restored best weights, learning history, and summary metrics.

def fit(
    model,
    train_loader,
    valid_loader,
    optimizer,
    *,
    epochs,
    use_augmentation,
    patience=6,
):
    history = {"train_loss": [], "valid_loss": [], "valid_accuracy": []}
    best_state = deepcopy(model.state_dict())
    best_loss = float("inf")
    best_epoch = 0
    stale = 0
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, _ = train_one_epoch(
            model,
            train_loader,
            optimizer,
            use_augmentation=use_augmentation
        )
        valid = evaluate(model, valid_loader)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid["loss"])
        history["valid_accuracy"].append(valid["accuracy"])

        if valid["loss"] < best_loss - 1e-4:
            best_loss = valid["loss"]
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    best_valid = evaluate(model, valid_loader)
    return {
        "history": history,
        "best_epoch": best_epoch,
        "valid_loss": best_valid["loss"],
        "valid_accuracy": best_valid["accuracy"],
        "seconds": time.perf_counter() - start,
    }


In [40]:
# Experiment using ResNet18

NUM_CLASSES = 7

weights = ResNet18_Weights.DEFAULT

frozen_model = resnet18(
    weights=weights
)

# Replace ImageNet classifier
num_features = frozen_model.fc.in_features

frozen_model.fc = nn.Linear(
    num_features,
    NUM_CLASSES
)

# Freeze everything
for param in frozen_model.parameters():
    param.requires_grad = False

# Unfreeze classification head
# Keep only the new classification head trainable
for param in frozen_model.fc.parameters():
    param.requires_grad = True

frozen_model = frozen_model.to(DEVICE)

# Only optimize the new classifier
frozen_optimizer = optim.Adam(
    frozen_model.fc.parameters(),
    lr=0.001
)

In [41]:
images, labels = next(iter(train_loader))

images = images.to(DEVICE)

outputs = frozen_model(images)

print("Input:", images.shape)
print("Output:", outputs.shape)

Input: torch.Size([16, 3, 160, 160])
Output: torch.Size([16, 7])


In [43]:
# Train

frozen_result = fit(
    frozen_model,
    train_loader,
    valid_loader,
    frozen_optimizer,
    epochs=10,
    use_augmentation=True,
    patience=3
)

In [44]:
final_frozen = evaluate(
    frozen_model,
    test_loader
)

print("\n========== FROZEN BACKBONE ==========")

print(
    f"Best epoch: "
    f"{frozen_result['best_epoch']}"
)

print(
    f"Validation accuracy: "
    f"{frozen_result['valid_accuracy']:.3f}"
)

print(
    f"Test accuracy: "
    f"{final_frozen['accuracy']:.3f}"
)

print(
    f"Training time: "
    f"{frozen_result['seconds']:.1f} seconds"
)


========== FROZEN BACKBONE ==========
Best epoch: 8
Validation accuracy: 0.951
Test accuracy: 0.933
Training time: 4989.9 seconds


In [ ]:
# partial fine tuning is not done due to time constraint on cpu